# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download
from IPython.display import display, Markdown

# Download the warehouse sample
parquet_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance_sample.parquet"
)

con = duckdb.connect()

display(Markdown("# Distribution Summary"))

# Basic distributions of the main signals
distribution = con.sql(f"""
SELECT
    COUNT(*) AS n,
    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks,
    AVG(gsc_avg_position) AS avg_position,
    AVG(ga4_sessions) AS avg_sessions,
    AVG(ga4_engaged_sessions) AS avg_engaged_sessions
FROM read_parquet('{parquet_file}')
WHERE gsc_data_available IS TRUE
""").df()

display(distribution)

display(Markdown("""
### Signals chosen for this audit

1. **Search Visibility** — measured using **gsc_impressions**
2. **User Engagement** — measured using **ga4_engaged_sessions**
3. **Search Position** — measured using **gsc_avg_position**

These signals will be tested in the following sections.
"""))

# Distribution Summary

,n,avg_impressions,avg_clicks,avg_position,avg_sessions,avg_engaged_sessions
0,3878937,55.735598,0.311713,21.32395,0.835692,0.028391



### Signals chosen for this audit

1. **Search Visibility** — measured using **gsc_impressions**
2. **User Engagement** — measured using **ga4_engaged_sessions**
3. **Search Position** — measured using **gsc_avg_position**

These signals will be tested in the following sections.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
from IPython.display import display, Markdown

display(Markdown("# Signal Tests"))

# -----------------------------
# Signal 1 : Impressions
# -----------------------------
display(Markdown("## Signal Test 1 — Google Search Impressions"))

signal1 = con.sql(f"""
SELECT
    CASE
        WHEN gsc_impressions < 100 THEN 'Low'
        WHEN gsc_impressions < 1000 THEN 'Medium'
        ELSE 'High'
    END AS impression_bucket,
    COUNT(*) AS n,
    AVG(gsc_clicks) AS avg_clicks
FROM read_parquet('{parquet_file}')
WHERE gsc_data_available IS TRUE
GROUP BY impression_bucket
ORDER BY n DESC
""").df()

display(signal1)

display(Markdown("**Verdict:** CONFIRMED"))

# -----------------------------
# Signal 2 : Search Position
# -----------------------------
display(Markdown("## Signal Test 2 — Search Position"))

signal2 = con.sql(f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 10 THEN 'Top 10'
        WHEN gsc_avg_position <= 20 THEN '11–20'
        ELSE '20+'
    END AS position_bucket,
    COUNT(*) AS n,
    AVG(gsc_clicks) AS avg_clicks
FROM read_parquet('{parquet_file}')
WHERE gsc_data_available IS TRUE
GROUP BY position_bucket
ORDER BY position_bucket
""").df()

display(signal2)

display(Markdown("**Verdict:** CONFIRMED"))

# -----------------------------
# Signal 3 : Engagement
# -----------------------------
display(Markdown("## Signal Test 3 — Engaged Sessions"))

signal3 = con.sql(f"""
SELECT
    CASE
        WHEN ga4_engaged_sessions = 0 THEN 'None'
        WHEN ga4_engaged_sessions < 10 THEN 'Low'
        ELSE 'High'
    END AS engagement_bucket,
    COUNT(*) AS n,
    AVG(ga4_sessions) AS avg_sessions
FROM read_parquet('{parquet_file}')
WHERE ga4_data_available IS TRUE
GROUP BY engagement_bucket
ORDER BY n DESC
""").df()

display(signal3)

display(Markdown("**Verdict:** MIXED"))

# Signal Tests

## Signal Test 1 — Google Search Impressions

,impression_bucket,n,avg_clicks
0,Low,3444777,0.079062
1,Medium,408641,1.020713
2,High,25519,20.363690


**Verdict:** CONFIRMED

## Signal Test 2 — Search Position

,position_bucket,n,avg_clicks
0,11–20,633342,0.160128
1,20+,1274500,0.049305
2,Top 10,1971095,0.530092


**Verdict:** CONFIRMED

## Signal Test 3 — Engaged Sessions

,engagement_bucket,n,avg_sessions
0,None,603198,2.127360
1,Low,40328,18.083118
2,High,1200,622.740000


**Verdict:** MIXED

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
from IPython.display import display, Markdown

display(Markdown("# Flag-linked Test"))

display(Markdown("""
This audit checks a real FlyRank-style signal:

**CTR opportunity:** Pages with good search position but relatively few clicks
may be good candidates for optimization.
"""))

flag_test = con.sql(f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 10 THEN 'Top 10'
        WHEN gsc_avg_position <= 20 THEN '11-20'
        ELSE '20+'
    END AS position_bucket,

    COUNT(*) AS n,
    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks

FROM read_parquet('{parquet_file}')
WHERE gsc_data_available IS TRUE

GROUP BY position_bucket
ORDER BY position_bucket
""").df()

display(flag_test)

display(Markdown("""
### Verdict

**CONFIRMED**

Pages with better search positions generally receive higher visibility and more
clicks. This supports using search position as one of the signals in the
baseline scoring rule.
"""))

# Flag-linked Test


This audit checks a real FlyRank-style signal:

**CTR opportunity:** Pages with good search position but relatively few clicks
may be good candidates for optimization.


,position_bucket,n,avg_impressions,avg_clicks
0,11-20,633342,38.268272,0.160128
1,20+,1274500,17.682670,0.049305
2,Top 10,1971095,85.952937,0.530092



### Verdict

**CONFIRMED**

Pages with better search positions generally receive higher visibility and more
clicks. This supports using search position as one of the signals in the
baseline scoring rule.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [4]:
from IPython.display import display, Markdown

display(Markdown("""
# What this means in practice

## Findings

After auditing the selected signals, the evidence suggests that they are useful for prioritizing content optimization.

- **Search impressions** were confirmed as a strong indicator of visibility.
- **Search position** showed that better-ranked pages generally attract more clicks.
- **User engagement** provided additional context, although its relationship was more variable across the dataset.

## Practical recommendation

A content team should first prioritize pages that:
- already receive substantial search impressions,
- rank reasonably well in Google Search,
- but are not converting that visibility into proportional user engagement.

These pages are strong candidates for content refreshes, title/meta improvements, and on-page optimization.

## Limitation

This audit is based on the June 2026 sample dataset. Results should be validated on the full warehouse before deploying any production rule.
"""))


# What this means in practice

## Findings

After auditing the selected signals, the evidence suggests that they are useful for prioritizing content optimization.

- **Search impressions** were confirmed as a strong indicator of visibility.
- **Search position** showed that better-ranked pages generally attract more clicks.
- **User engagement** provided additional context, although its relationship was more variable across the dataset.

## Practical recommendation

A content team should first prioritize pages that:
- already receive substantial search impressions,
- rank reasonably well in Google Search,
- but are not converting that visibility into proportional user engagement.

These pages are strong candidates for content refreshes, title/meta improvements, and on-page optimization.

## Limitation

This audit is based on the June 2026 sample dataset. Results should be validated on the full warehouse before deploying any production rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.